# JetRacer — Object Detection with Fixed Movement Scheme (OpenCV DNN, host-only)

This version runs entirely in your normal JetRacer Jupyter Lab (port 8888) — no Docker, no `jetson_inference`. It uses OpenCV's built-in DNN module with SSD-MobileNet-v2 trained on COCO (80 classes, including stop sign, traffic light, person, car, and more).

This notebook:
1. Downloads the (small, one-time) detection model files
2. Reads frames from the camera
3. Runs object detection each frame
4. Maps the detected class to a fixed movement, and drives the car
5. Shows a live preview with detection boxes drawn on it

Run the cells in order top to bottom.

## 1. Download the detection model (one-time)

This pulls SSD-MobileNet-v2 trained on COCO (80 real-world classes, including `stop sign`, `person`, `car`, `bottle`, `chair`, and more). It's a bigger download than the previous VOC model (~65MB), so this cell can take a minute or two depending on your connection. `-nc` means "no clobber" — safe to re-run, it won't re-download if the files already exist.

In [ ]:
!wget -nc http://download.tensorflow.org/models/object_detection/ssd_mobilenet_v2_coco_2018_03_29.tar.gz
!tar -xzf ssd_mobilenet_v2_coco_2018_03_29.tar.gz
!wget -nc https://raw.githubusercontent.com/opencv/opencv_extra/master/testdata/dnn/ssd_mobilenet_v2_coco_2018_03_29.pbtxt

## 2. Imports and camera setup

If another notebook still has the camera open, this cell will fail. Shut down other running kernels first (File → Hub Control Panel, or the 'Running Terminals and Kernels' tab).

In [ ]:
import time
import cv2
import numpy as np
import ipywidgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar

camera = CSICamera(width=224, height=224, capture_fps=30)
print("Camera initialized.")

## 3. Load the detection model

COCO's 80 classes span everyday objects: people, vehicles, road items (stop sign, traffic light), animals, furniture, and household objects. The IDs below have gaps (e.g. no 12, 26, 29) because that's how the original COCO label map is numbered — this is normal, not a mistake.

In [ ]:
net = cv2.dnn.readNetFromTensorflow(
    "ssd_mobilenet_v2_coco_2018_03_29/frozen_inference_graph.pb",
    "ssd_mobilenet_v2_coco_2018_03_29.pbtxt"
)

COCO_CLASSES = {
    1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane", 6: "bus",
    7: "train", 8: "truck", 9: "boat", 10: "traffic light", 11: "fire hydrant",
    13: "stop sign", 14: "parking meter", 15: "bench", 16: "bird", 17: "cat",
    18: "dog", 19: "horse", 20: "sheep", 21: "cow", 22: "elephant", 23: "bear",
    24: "zebra", 25: "giraffe", 27: "backpack", 28: "umbrella", 31: "handbag",
    32: "tie", 33: "suitcase", 34: "frisbee", 35: "skis", 36: "snowboard",
    37: "sports ball", 38: "kite", 39: "baseball bat", 40: "baseball glove",
    41: "skateboard", 42: "surfboard", 43: "tennis racket", 44: "bottle",
    46: "wine glass", 47: "cup", 48: "fork", 49: "knife", 50: "spoon",
    51: "bowl", 52: "banana", 53: "apple", 54: "sandwich", 55: "orange",
    56: "broccoli", 57: "carrot", 58: "hot dog", 59: "pizza", 60: "donut",
    61: "cake", 62: "chair", 63: "couch", 64: "potted plant", 65: "bed",
    67: "dining table", 70: "toilet", 72: "tv", 73: "laptop", 74: "mouse",
    75: "remote", 76: "keyboard", 77: "cell phone", 78: "microwave", 79: "oven",
    80: "toaster", 81: "sink", 82: "refrigerator", 84: "book", 85: "clock",
    86: "vase", 87: "scissors", 88: "teddy bear", 89: "hair drier", 90: "toothbrush",
}

CONFIDENCE_THRESHOLD = 0.5
print("Model loaded.")

## 4. Car setup

In [ ]:
car = NvidiaRacecar()
car.throttle = 0.0
car.steering = 0.0
print("Car initialized.")

## 5. Movement actions and class-to-action mapping

Edit the `ACTIONS` dict to change which class triggers which movement, and edit the durations/steering/throttle values to tune the maneuver. Class names must exactly match strings in `COCO_CLASSES` above (e.g. `"stop sign"`, `"person"`, `"bottle"`, `"chair"`).

In [ ]:
def stop_car():
    car.throttle = 0.0
    car.steering = 0.0

def turn_left():
    car.steering = -0.5
    car.throttle = 0.2
    time.sleep(1.0)
    stop_car()

def turn_right():
    car.steering = 0.5
    car.throttle = 0.2
    time.sleep(1.0)
    stop_car()

def drive_forward():
    car.steering = 0.0
    car.throttle = 0.25
    time.sleep(1.0)
    stop_car()

# Map detected class label -> action function.
# Edit these to match the classes you actually care about (must match names in COCO_CLASSES above).
ACTIONS = {
    "stop sign": stop_car,
    "person": turn_left,
    "bottle": turn_right,
}

COOLDOWN_SECONDS = 2.0  # minimum time between triggered actions, to avoid re-triggering every frame

## 6. Live preview widget

Run this cell once before the main loop — the preview will appear right below it and update live once the loop is running.

In [ ]:
image_widget = ipywidgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

PREVIEW_EVERY_N_FRAMES = 3  # only push every Nth frame to the widget, to limit overhead on the Nano

## 7. Main loop

Use the Jupyter stop button (square icon) to interrupt this cell. It will fall through to the `except KeyboardInterrupt` block and stop the car safely rather than leaving it mid-maneuver.

Scroll back up to the image widget above (Cell 6's output) to watch the live feed with detection boxes while this runs.

In [ ]:
last_action_time = 0.0
frame_count = 0

try:
    while True:
        frame = camera.read()
        (h, w) = frame.shape[:2]

        blob = cv2.dnn.blobFromImage(frame, size=(300, 300), swapRB=True, crop=False)
        net.setInput(blob)
        detections = net.forward()

        best_label = None
        best_confidence = 0.0
        best_box = None

        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > CONFIDENCE_THRESHOLD and confidence > best_confidence:
                class_id = int(detections[0, 0, i, 1])
                if class_id not in COCO_CLASSES:
                    continue
                best_label = COCO_CLASSES[class_id]
                best_confidence = confidence
                box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
                best_box = box.astype("int")

        now = time.time()
        if best_label is not None:
            if best_label in ACTIONS and (now - last_action_time) > COOLDOWN_SECONDS:
                print(f"Detected: {best_label} (confidence {best_confidence:.2f}) -> {ACTIONS[best_label].__name__}")
                ACTIONS[best_label]()
                last_action_time = now
        else:
            stop_car()

        # Draw the box/label onto the frame and push it to the widget every N frames
        if frame_count % PREVIEW_EVERY_N_FRAMES == 0:
            display_frame = frame.copy()
            if best_box is not None:
                (startX, startY, endX, endY) = best_box
                cv2.rectangle(display_frame, (startX, startY), (endX, endY), (0, 255, 0), 2)
                cv2.putText(display_frame, f"{best_label}: {best_confidence:.2f}", (startX, max(startY - 10, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            image_widget.value = bytes(cv2.imencode('.jpg', display_frame)[1])

        frame_count += 1

except KeyboardInterrupt:
    print("Interrupted, stopping car.")
    stop_car()

## 8. Cleanup

Run this before closing the notebook or opening a different camera/car notebook, so the camera and motors are released cleanly.

In [ ]:
stop_car()
camera.running = False
print("Stopped and released camera.")